In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sympy import pretty_print
from torch.utils.data import Dataset



In [66]:
class LSTM_DataSet(Dataset):
    def __init__(self, df,window_size,features,targets):
        self.df = df
        self.window_size = window_size
        self.X = []
        self.Y = []
        self.features = features
        self.targets = targets

# self.X = [
#     match_1 = [
#         window_1 = [frame_1_features, frame_2_features, frame_3_features],
#         window_2 = [...],
#         ...
#     ],
#     match_2 = [...],
#     ...
# ]

    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        X_tensor = torch.tensor(self.X[idx], dtype=torch.float32)
        Y_tensor = torch.tensor(self.Y[idx], dtype=torch.float32)
        return X_tensor, Y_tensor


    def get_match_ids(self):
        return self.df["match_id"].unique().tolist()

    def transform(self):
        match_ids = self.get_match_ids()
        self.X = []  # List of match windows
        self.Y = []  # List of match targets
        self.X = []  # List of match windows
        self.Y = []  # List of match targets

        for idx in match_ids:
            match_set = self.df[self.df["match_id"] == idx]
            match_X = []
            match_Y = []

            for i in range(len(match_set)):
                start = max(0, i - self.window_size)
                if i == 0:
                    window_frames = []
                else:
                    window_frames = match_set.iloc[start:i][self.features].values.tolist()

                if len(window_frames) < self.window_size:
                    pad_rows = self.window_size - len(window_frames)
                    padding = [[0] * len(self.features)] * pad_rows
                    window_frames = padding + window_frames

                target_frames = match_set.iloc[i][self.targets].values.tolist()

                match_X.append(window_frames)
                match_Y.append(target_frames)

            self.X.append(match_X)
            self.Y.append(match_Y)


In [67]:
full_df = pd.read_csv('full_cleaned_dataset_lstm.csv')
features = [col for col in full_df.columns if col not in ['match_id',"id" ,'action', 'pos_x', 'pos_y']]
targets = ['action', 'pos_x', 'pos_y']

In [68]:
dataset = LSTM_DataSet(full_df, window_size=3, features=features, targets=targets)
dataset.transform()

In [69]:
train_transformed = LSTM_DataSet(full_df, window_size=10, features=features, targets=targets)
train_transformed.transform()
test_transformed = LSTM_DataSet(full_df, window_size=10, features=features, targets=targets)
test_transformed.transform()

In [75]:
X = np.vstack([np.array(match) for match in train_transformed.X])
Y = np.concatenate([np.array(match) for match in test_transformed.Y])

array([ 0., -1., -1.])

In [76]:
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)